# 02 – Park-specific XGBoost Modeling

Trains a park-specific XGBoost classifier for each of the 35 Japanese national parks
using spatially grouped nested cross-validation with a k-ring gap to reduce spatial leakage.
After cross-validation, each model is refit on the full balanced dataset and used
to generate a nationwide environmental similarity surface.

Key stability features:
- Block-matched negative sampling with global mixing (`mix_global_frac=0.3`)
- Automatic `group_cut` tuning to ensure sufficient positive spatial groups
- Greedy fold assignment with single-class repair
- Automatic gap relaxation per fold

**Input**  : `data/interim/h3_jpn_res9_processed.parquet`
**Output** : `data/results/{park_slug}_prod_g7_gap1/` (per-park model, scores, metadata)

Corresponds to *Section 2.3 – Model development* and *Section 2.4 – Nationwide extrapolation*
in the manuscript.

## Imports and configuration

In [1]:
import hashlib
import json
import joblib
import os
import random
import warnings
from collections import Counter
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.metrics import (
    accuracy_score, average_precision_score, f1_score,
    precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import shuffle
from xgboost import XGBClassifier

from config import (
    DATA_DIR, OUTPUT_DIR, PARKS,
    GLOBAL_SEED, GROUP_CUT, GAP_KRING, OUTER_SPLITS, INNER_SPLITS,
)

warnings.filterwarnings(
    "ignore", category=UndefinedMetricWarning,
    message="No positive class found in y_true.*"
)

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load processed dataset

In [2]:
df_all = pd.read_parquet(DATA_DIR / "h3_jpn_res9_processed.parquet")
print(f"Loaded: {df_all.shape[0]:,} rows x {df_all.shape[1]} columns")

Loaded: 4,051,335 rows x 42 columns


## Utility functions

In [3]:
# ── Basic helpers ──────────────────────────────────────────────────────────────

def _hash_list(lst: List[str]) -> str:
    return hashlib.sha256(("\n".join(sorted(lst))).encode()).hexdigest()[:12]

def _has_both_classes(y: np.ndarray) -> bool:
    return bool(np.any(y == 1) and np.any(y == 0))


# ── Threshold utilities ────────────────────────────────────────────────────────

def find_optimal_threshold(
    y_true: np.ndarray, y_score: np.ndarray, method: str = "f1"
) -> float:
    if method == "f1":
        precision, recall, thresholds = precision_recall_curve(y_true, y_score)
        f1s = (2 * precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-12)
        return 0.5 if len(f1s) == 0 else float(thresholds[int(np.nanargmax(f1s))])
    elif method == "youden":
        fpr, tpr, thresholds = roc_curve(y_true, y_score)
        j = tpr - fpr
        return 0.5 if len(j) == 0 else float(thresholds[int(np.nanargmax(j))])
    else:
        raise ValueError("method must be 'f1' or 'youden'")

def find_threshold_f1_constrained(
    y_true, y_score, min_precision=None, min_recall=None
) -> float:
    p, r, t = precision_recall_curve(y_true, y_score)
    f1 = (2 * p[:-1] * r[:-1]) / (p[:-1] + r[:-1] + 1e-12)
    mask = np.ones_like(f1, dtype=bool)
    if min_precision is not None:
        mask &= (p[:-1] >= float(min_precision))
    if min_recall is not None:
        mask &= (r[:-1] >= float(min_recall))
    if mask.any():
        return float(t[int(np.nanargmax(np.where(mask, f1, -np.inf)))])
    return find_optimal_threshold(y_true, y_score, method="f1")


# ── H3 k-ring gap support ──────────────────────────────────────────────────────

try:
    import h3
    HAVE_H3 = True
except Exception:
    HAVE_H3 = False

def _kring_one(h, k):
    try:
        return set(h3.k_ring(h, k))
    except Exception:
        try:
            return set(h3.grid_disk(h, k))
        except Exception:
            return {h}

def expand_by_k_ring(h3_list, k):
    if not HAVE_H3 or k <= 0:
        return set(map(str, h3_list))
    out = set()
    for h in h3_list:
        out |= _kring_one(str(h), k)
    return out


# ── Block-matched negative sampling ───────────────────────────────────────────

def sample_negatives_block_matched(
    df_all: pd.DataFrame,
    df_positive: pd.DataFrame,
    group_cut: int,
    n_negative: int,
    seed: int = 42,
    mix_global_frac: float = 0.3,
    allow_replacement: bool = False,
) -> pd.DataFrame:
    """
    Sample negatives that spatially match the distribution of positives
    (block-matched), with a fraction of global random negatives mixed in.
    mix_global_frac=0.3 means 30% global random, 70% block-matched.
    """

    pos = df_positive.copy()
    pos["__g"] = pos["h3_9"].apply(lambda c: h3.cell_to_parent(c, group_cut))

    rng = np.random.RandomState(seed)
    neg_pool = df_all[df_all["np_class"] == 0].copy()
    neg_pool["__g"] = neg_pool["h3_9"].apply(lambda c: h3.cell_to_parent(c, group_cut))

    n_global = int(round(n_negative * float(mix_global_frac)))
    n_local  = n_negative - n_global

    # (A) Block-matched local negatives
    need = pos["__g"].value_counts().to_dict()
    neg_parts = []
    used_idx: set = set()

    for g, k in sorted(need.items(), key=lambda kv: kv[1], reverse=True):
        cand = neg_pool[neg_pool["__g"] == g]
        if cand.empty:
            continue
        take = min(k, len(cand)) if not allow_replacement else k
        samp = cand.sample(
            n=take, random_state=int(rng.randint(0, 1_000_000)),
            replace=allow_replacement,
        )
        if not allow_replacement:
            samp = samp.loc[~samp.index.isin(used_idx)]
            used_idx |= set(samp.index)
        neg_parts.append(samp)

    neg_local = pd.concat(neg_parts, axis=0) if neg_parts else neg_pool.head(0)

    # Fill remaining local quota from rest of pool
    remain_local = n_local - len(neg_local)
    if remain_local > 0:
        rest_pool = neg_pool if allow_replacement else neg_pool.loc[~neg_pool.index.isin(used_idx)]
        extra = rest_pool.sample(
            n=remain_local,
            random_state=int(rng.randint(0, 1_000_000)),
            replace=(len(rest_pool) < remain_local),
        )
        neg_local = pd.concat([neg_local, extra], axis=0)

    # (B) Global random negatives
    if n_global > 0:
        global_pool = df_all[df_all["np_class"] == 0]
        neg_global = global_pool.sample(
            n=n_global,
            random_state=int(rng.randint(0, 1_000_000)),
            replace=(len(global_pool) < n_global),
        )
        df_negative = pd.concat([neg_local, neg_global], axis=0)
    else:
        df_negative = neg_local

    return df_negative.drop(columns=["__g"], errors="ignore")


# ── Spatial fold assignment and repair ────────────────────────────────────────

def build_group_stats(groups: np.ndarray, y: np.ndarray):
    uniq = np.unique(groups)
    pos  = {g: int((y[groups == g] == 1).sum()) for g in uniq}
    neg  = {g: int((y[groups == g] == 0).sum()) for g in uniq}
    size = {g: int((groups == g).sum())          for g in uniq}
    return uniq, pos, neg, size

def greedy_assign_groups_to_folds(
    groups: np.ndarray, y: np.ndarray, n_splits: int, seed: int = 42
) -> List[set]:
    """Assign spatial groups to folds balancing positive counts (greedy)."""
    uniq, pos, neg, size = build_group_stats(groups, y)
    order = sorted(uniq, key=lambda g: (pos[g], size[g]), reverse=True)
    folds: List[set] = [set() for _ in range(n_splits)]
    fold_pos = [0] * n_splits
    for g in order:
        i = int(np.argmin(fold_pos))
        folds[i].add(g)
        fold_pos[i] += pos[g]
    return folds

def repair_folds_to_avoid_single_class_test(
    groups: np.ndarray, y: np.ndarray, folds: List[set], max_moves: int = 200
) -> List[set]:
    """Move groups between folds so every test fold has both classes."""
    uniq, pos, neg, size = build_group_stats(groups, y)

    def fold_counts(fi):
        gs = folds[fi]
        return sum(pos[g] for g in gs), sum(neg[g] for g in gs)

    moves = 0
    while moves < max_moves:
        bad_folds = [fi for fi in range(len(folds)) if 0 in fold_counts(fi)]
        if not bad_folds:
            break
        fixed_any = False
        for fi in bad_folds:
            p, n = fold_counts(fi)
            need_pos, need_neg = (p == 0), (n == 0)
            best_move = None
            for dj in range(len(folds)):
                if dj == fi:
                    continue
                for g in list(folds[dj]):
                    if need_pos and pos[g] == 0:
                        continue
                    if need_neg and neg[g] == 0:
                        continue
                    donor_after = folds[dj] - {g}
                    if not donor_after:
                        continue
                    dp = sum(pos[x] for x in donor_after)
                    dn = sum(neg[x] for x in donor_after)
                    if dp == 0 or dn == 0:
                        continue
                    gain = (10 if need_pos and pos[g] > 0 else 0) + \
                           (10 if need_neg and neg[g] > 0 else 0)
                    score = gain - 0.001 * size[g]
                    if best_move is None or score > best_move[0]:
                        best_move = (score, g, dj, fi)
            if best_move is not None:
                _, g, dj, fi2 = best_move
                folds[dj].remove(g)
                folds[fi2].add(g)
                moves += 1
                fixed_any = True
        if not fixed_any:
            break
    return folds

def folds_to_indices(groups: np.ndarray, folds: List[set]):
    for i in range(len(folds)):
        te_mask = np.isin(groups, list(folds[i]))
        yield np.where(~te_mask)[0], np.where(te_mask)[0]


# ── Automatic group_cut tuning ─────────────────────────────────────────────────

def choose_group_cut_auto(
    df_sampled: pd.DataFrame,
    base_group_cut: int,
    min_pos_groups: int,
    try_range: Tuple[int, int] = (5, 9),
) -> int:
    """Find the shortest H3 prefix that yields at least min_pos_groups positive blocks."""
    best, best_n = base_group_cut, 0
    for cut in range(try_range[0], try_range[1] + 1):
        g = df_sampled["h3_9"].apply(lambda c: h3.cell_to_parent(c, cut)).values
        n = len(np.unique(g[df_sampled["np_class"].astype(int).values == 1]))
        if n >= min_pos_groups:
            return cut
        if n > best_n:
            best_n, best = n, cut
    return best

## Park-specific modeling pipeline

The function below encapsulates the full workflow for a single park:

1. Block-matched balanced sampling (positive spatial distribution preserved)
2. Automatic `group_cut` tuning to ensure sufficient positive spatial blocks
3. Greedy fold assignment with single-class repair
4. Spatially grouped nested cross-validation with automatic gap relaxation
5. Hyperparameter selection (inner CV, PR-AUC criterion)
6. CV-averaged threshold calibration
7. Full-data refit and nationwide similarity extrapolation

See *Section 2.3* in the manuscript for methodological details.

In [4]:
def process_park_pipeline(
    df_all: pd.DataFrame,
    park_name: str,
    output_dir: str,
    outer_splits: int = OUTER_SPLITS,
    inner_splits: int = INNER_SPLITS,
    threshold_method: str = "f1",
    param_grid: Optional[List[Dict[str, Any]]] = None,
    label_shuffle: bool = False,
    group_cut: int = GROUP_CUT,
    gap_kring: int = GAP_KRING,
    gap_prefix_cut: Optional[int] = None,
    threshold_source: str = "cv",
    mix_global_frac: float = 0.3,
    auto_group_cut: bool = True,
    auto_gap_relax: bool = True,
    gap_relax_steps: Tuple[int, ...] = (1, 0),
    max_fold_repairs: int = 200,
    min_precision: Optional[float] = None,
    min_recall: Optional[float] = None,
) -> Dict[str, Any]:
    """
    Train, evaluate, and extrapolate a park-specific XGBoost classifier.

    Parameters
    ----------
    park_name : str
        English park slug matching the NAME column after preprocessing.
    mix_global_frac : float
        Fraction of negatives sampled globally (rest are block-matched).
    auto_group_cut : bool
        Automatically adjust group_cut to ensure enough positive spatial groups.
    auto_gap_relax : bool
        Per fold, try gap_relax_steps in order; use first that keeps both classes.
    label_shuffle : bool
        If True, shuffle labels before training (negative control).
    """
    os.makedirs(output_dir, exist_ok=True)

    # 1. Positive samples
    df_positive = df_all[(df_all["NAME"] == park_name) & (df_all["np_class"] == 1)]
    if len(df_positive) == 0:
        raise ValueError(f"No positive samples for park: {park_name}")

    # 2. Block-matched negative sampling
    df_negative = sample_negatives_block_matched(
        df_all=df_all,
        df_positive=df_positive,
        group_cut=group_cut,
        n_negative=len(df_positive),
        seed=GLOBAL_SEED,
        mix_global_frac=mix_global_frac,
        allow_replacement=False,
    )

    df_sampled = (
        pd.concat([df_positive, df_negative])
        .sample(frac=1.0, random_state=GLOBAL_SEED)
        .reset_index(drop=True)
    )

    drop_columns = ["h3_9", "NAME", "ZONE", "np_class", "lithology1", "h3int"]
    feature_columns = [c for c in df_sampled.columns if c not in drop_columns]

    X = df_sampled[feature_columns].copy()
    y = df_sampled["np_class"].astype(int).values

    if label_shuffle:
        y = shuffle(y, random_state=GLOBAL_SEED)

    # 3. Automatic group_cut tuning
    base_group_cut = group_cut
    if auto_group_cut:
        group_cut = choose_group_cut_auto(
            df_sampled, base_group_cut,
            min_pos_groups=max(2, outer_splits),
            try_range=(5, 9),
        )

    group_labels = df_sampled["h3_9"].apply(lambda c: h3.cell_to_parent(c, group_cut)).values
    pos_groups = np.unique(group_labels[y == 1])
    eff_outer_splits = min(outer_splits, max(2, len(pos_groups)))
    if eff_outer_splits < outer_splits:
        print(f"[{park_name}] outer_splits reduced {outer_splits} -> {eff_outer_splits} "
              f"(positive groups={len(pos_groups)})")

    # 4. Hyperparameter candidates
    if param_grid is None:
        param_grid = [
            {"max_depth": 4, "learning_rate": 0.1,  "n_estimators": 300,
             "subsample": 0.8, "colsample_bytree": 0.8},
            {"max_depth": 6, "learning_rate": 0.1,  "n_estimators": 400,
             "subsample": 0.8, "colsample_bytree": 0.8},
            {"max_depth": 6, "learning_rate": 0.05, "n_estimators": 600,
             "subsample": 0.9, "colsample_bytree": 0.9},
        ]

    # 5. Build folds: greedy assignment + single-class repair
    folds = greedy_assign_groups_to_folds(group_labels, y, eff_outer_splits, seed=GLOBAL_SEED)
    folds = repair_folds_to_avoid_single_class_test(
        group_labels, y, folds, max_moves=max_fold_repairs
    )

    outer_metrics: List[Dict[str, Any]] = []
    outer_thresholds: List[float] = []
    chosen_params_each_fold: List[Dict[str, Any]] = []
    skipped_folds = 0
    oof_scores, oof_truth = [], []

    # Helper: apply k-ring gap with fallback to prefix matching
    def apply_gap(tr_idx: np.ndarray, te_idx: np.ndarray, gap_k: int) -> np.ndarray:
        if gap_k is None or (gap_k <= 0 and gap_prefix_cut is None):
            return tr_idx
        orig = tr_idx.copy()
        test_h3s = df_sampled.iloc[te_idx]["h3_9"].astype(str).tolist()
        if HAVE_H3 and gap_k > 0:
            gap_set = expand_by_k_ring(test_h3s, k=gap_k)
            keep = ~np.isin(
                df_sampled.iloc[tr_idx]["h3_9"].astype(str).values,
                list(gap_set)
            )
            tr_idx2 = tr_idx[keep]
        else:
            cut = gap_prefix_cut if gap_prefix_cut is not None else group_cut
            parent_set = {h3.cell_to_parent(h, cut) for h in test_h3s}
            keep = ~df_sampled.iloc[tr_idx]["h3_9"].apply(lambda c: h3.cell_to_parent(c, cut)).isin(parent_set)
            tr_idx2 = tr_idx[keep.values]
        ok = (len(tr_idx2) >= 10) and (len(np.unique(y[tr_idx2])) == 2)
        return tr_idx2 if ok else orig

    # 6. Outer CV loop
    for fold_idx, (tr_idx, te_idx) in enumerate(
        folds_to_indices(group_labels, folds), start=1
    ):
        tr_idx = np.array(tr_idx)
        te_idx = np.array(te_idx)

        # Gap with automatic per-fold relaxation
        used_gap = gap_kring
        if auto_gap_relax:
            for gk in gap_relax_steps:
                tr_try = apply_gap(tr_idx, te_idx, gk)
                if (len(tr_try) >= 10) and (len(np.unique(y[tr_try])) == 2):
                    tr_idx, used_gap = tr_try, gk
                    break
        else:
            tr_idx = apply_gap(tr_idx, te_idx, gap_kring)

        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        if not _has_both_classes(y_te):
            skipped_folds += 1
            print(f"[Fold {fold_idx}] Single-class test set - skipped")
            continue

        # Inner CV: select hyperparameters by PR-AUC
        min_class = min(int((y_tr == 0).sum()), int((y_tr == 1).sum()))
        inner_splits_eff = max(2, min(inner_splits, min_class))
        inner_cv = StratifiedKFold(
            n_splits=inner_splits_eff, shuffle=True, random_state=GLOBAL_SEED
        )

        best_score, best_params, best_inner_threshold = -np.inf, None, None

        for p in param_grid:
            inner_scores, inner_thresholds, valid = [], [], 0
            for inner_tr, inner_va in inner_cv.split(X_tr, y_tr):
                X_itr, X_iva = X_tr.iloc[inner_tr], X_tr.iloc[inner_va]
                y_itr, y_iva = y_tr[inner_tr], y_tr[inner_va]
                if not _has_both_classes(y_iva):
                    continue
                model = XGBClassifier(
                    **p, random_state=GLOBAL_SEED, eval_metric="logloss",
                    n_jobs=-1, tree_method="hist", verbosity=0,
                )
                model.fit(X_itr, y_itr)
                y_va_score = model.predict_proba(X_iva)[:, 1]
                inner_scores.append(average_precision_score(y_iva, y_va_score))
                inner_thresholds.append(
                    find_optimal_threshold(y_iva, y_va_score, method=threshold_method)
                )
                valid += 1
            if valid == 0:
                continue
            mean_score = float(np.mean(inner_scores))
            if mean_score > best_score:
                best_score = mean_score
                best_params = p
                best_inner_threshold = float(np.mean(inner_thresholds))

        if best_params is None:
            best_params = param_grid[0]
            best_inner_threshold = 0.5

        chosen_params_each_fold.append(best_params)

        # Outer evaluation
        final_model = XGBClassifier(
            **best_params, random_state=GLOBAL_SEED, eval_metric="logloss",
            n_jobs=-1, tree_method="hist", verbosity=0,
        )
        final_model.fit(X_tr, y_tr)

        y_score_te = final_model.predict_proba(X_te)[:, 1]
        y_pred_te  = (y_score_te >= best_inner_threshold).astype(int)

        outer_metrics.append({
            "fold":           fold_idx,
            "roc_auc":        roc_auc_score(y_te, y_score_te),
            "pr_auc":         average_precision_score(y_te, y_score_te),
            "f1":             f1_score(y_te, y_pred_te, zero_division=0),
            "accuracy":       accuracy_score(y_te, y_pred_te),
            "threshold":      float(best_inner_threshold),
            "test_pos":       int(y_te.sum()),
            "test_neg":       int(len(y_te) - int(y_te.sum())),
            "used_gap_kring": int(used_gap) if used_gap is not None else None,
        })
        outer_thresholds.append(float(best_inner_threshold))
        oof_scores.append(y_score_te)
        oof_truth.append(y_te)

    if len(outer_metrics) == 0:
        raise RuntimeError(
            f"No valid outer folds for {park_name}. "
            "Try increasing mix_global_frac or reducing outer_splits."
        )

    # Aggregate metrics
    def agg(key):
        vals = [m[key] for m in outer_metrics]
        return float(np.mean(vals)), float(np.std(vals))

    agg_results = {
        "roc_auc_mean": agg("roc_auc")[0], "roc_auc_std": agg("roc_auc")[1],
        "pr_auc_mean":  agg("pr_auc")[0],  "pr_auc_std":  agg("pr_auc")[1],
        "f1_mean":      agg("f1")[0],      "f1_std":      agg("f1")[1],
        "acc_mean":     agg("accuracy")[0], "acc_std":     agg("accuracy")[1],
        "threshold_mean_cv": float(np.mean(outer_thresholds)),
        "threshold_std_cv":  float(np.std(outer_thresholds)),
    }

    # OOF threshold
    oof_scores_arr = np.concatenate(oof_scores) if oof_scores else np.array([])
    oof_truth_arr  = np.concatenate(oof_truth)  if oof_truth  else np.array([])
    threshold_oof = None
    if oof_scores_arr.size > 0:
        if min_precision is not None or min_recall is not None:
            threshold_oof = find_threshold_f1_constrained(
                oof_truth_arr, oof_scores_arr,
                min_precision=min_precision, min_recall=min_recall,
            )
        else:
            threshold_oof = find_optimal_threshold(
                oof_truth_arr, oof_scores_arr, method=threshold_method
            )

    production_threshold = (
        float(threshold_oof)
        if threshold_source == "oof" and threshold_oof is not None
        else float(agg_results["threshold_mean_cv"])
    )

    # 7. Refit on full balanced dataset and extrapolate nationwide
    best_overall_params = chosen_params_each_fold[
        int(np.argmax([m["pr_auc"] for m in outer_metrics]))
    ]
    final_model_full = XGBClassifier(
        **best_overall_params, random_state=GLOBAL_SEED, eval_metric="logloss",
        n_jobs=-1, tree_method="hist", verbosity=0,
    )
    final_model_full.fit(X, y)

    df_nationwide = df_all.copy()
    df_nationwide["similarity_score"] = \
        final_model_full.predict_proba(df_nationwide[feature_columns])[:, 1]
    df_nationwide["similarity_label"] = \
        (df_nationwide["similarity_score"] >= production_threshold).astype(int)

    # Save artefacts
    model_id = _hash_list(feature_columns) + f"-{park_name}"
    joblib.dump(
        final_model_full,
        os.path.join(output_dir, f"{park_name}_xgb_model_cv.joblib"),
    )
    df_nationwide.to_parquet(
        os.path.join(output_dir, f"{park_name}_nationwide_similarity_cv.parquet"),
        index=False,
    )
    df_nationwide[["h3_9", "similarity_score", "similarity_label"]].to_csv(
        os.path.join(output_dir, f"{park_name}_similarity_scores_cv.csv"),
        index=False,
    )

    metadata = {
        "park_name":               park_name,
        "n_positive":              int((df_sampled["np_class"] == 1).sum()),
        "n_negative":              int((df_sampled["np_class"] == 0).sum()),
        "feature_count":           int(len(feature_columns)),
        "features":                feature_columns,
        "model_id":                model_id,
        "param_grid":              param_grid,
        "chosen_params_each_fold": chosen_params_each_fold,
        "outer_metrics":           outer_metrics,
        "aggregated":              agg_results,
        "production_threshold":    production_threshold,
        "threshold_source":        threshold_source,
        "threshold_oof":           threshold_oof,
        "random_seed":             GLOBAL_SEED,
        "label_shuffle":           label_shuffle,
        "group_cut_base":          int(base_group_cut),
        "group_cut_used":          int(group_cut),
        "gap_kring_requested":     int(gap_kring) if gap_kring is not None else None,
        "gap_prefix_cut":          gap_prefix_cut,
        "mix_global_frac":         float(mix_global_frac),
        "outer_splits_requested":  int(outer_splits),
        "outer_splits_effective":  int(eff_outer_splits),
        "skipped_folds":           int(skipped_folds),
        "auto_group_cut":          bool(auto_group_cut),
        "auto_gap_relax":          bool(auto_gap_relax),
        "gap_relax_steps":         list(gap_relax_steps),
        "max_fold_repairs":        int(max_fold_repairs),
        "min_precision":           min_precision,
        "min_recall":              min_recall,
    }
    with open(
        os.path.join(output_dir, f"{park_name}_metadata_cv.json"),
        "w", encoding="utf-8",
    ) as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print(
        f"[{park_name}] "
        f"PR-AUC={agg_results['pr_auc_mean']:.3f}  "
        f"ROC-AUC={agg_results['roc_auc_mean']:.3f}  "
        f"F1={agg_results['f1_mean']:.3f}  "
        f"threshold({threshold_source})={production_threshold:.3f}  "
        f"group_cut={group_cut}  gap={gap_kring}"
    )
    return metadata

## Run production models for all 35 parks

Each park is processed independently. Results are written to
`data/results/{slug}_prod_g7_gap1/`.

Runtime: approximately 2-4 hours on a modern laptop.

In [5]:
for slug in PARKS:
    outdir = OUTPUT_DIR / f"{slug}_prod_g7_gap1"
    try:
        process_park_pipeline(
            df_all=df_all,
            park_name=slug,
            output_dir=str(outdir),
            outer_splits=OUTER_SPLITS,
            inner_splits=INNER_SPLITS,
            threshold_method="f1",
            label_shuffle=False,
            group_cut=GROUP_CUT,
            gap_kring=GAP_KRING,
            threshold_source="cv",
            mix_global_frac=0.3,
            auto_group_cut=True,
            auto_gap_relax=True,
            gap_relax_steps=(1, 0),
        )
    except Exception as e:
        print(f"[ERROR] {slug}: {e}")

[rishiri] PR-AUC=0.787  ROC-AUC=0.633  F1=0.698  threshold(cv)=0.435  group_cut=5  gap=1
[shiretoko] PR-AUC=0.822  ROC-AUC=0.747  F1=0.692  threshold(cv)=0.412  group_cut=5  gap=1
[akan] PR-AUC=0.902  ROC-AUC=0.770  F1=0.751  threshold(cv)=0.556  group_cut=5  gap=1
[kushiro] PR-AUC=0.987  ROC-AUC=0.945  F1=0.924  threshold(cv)=0.605  group_cut=6  gap=1
[taisetsu] PR-AUC=0.974  ROC-AUC=0.858  F1=0.835  threshold(cv)=0.518  group_cut=5  gap=1
[hidaka] PR-AUC=0.977  ROC-AUC=0.942  F1=0.854  threshold(cv)=0.441  group_cut=5  gap=1
[shikotsu] PR-AUC=0.843  ROC-AUC=0.757  F1=0.768  threshold(cv)=0.501  group_cut=5  gap=1
[towada] PR-AUC=0.926  ROC-AUC=0.682  F1=0.766  threshold(cv)=0.545  group_cut=5  gap=1
[sanriku] PR-AUC=0.813  ROC-AUC=0.658  F1=0.777  threshold(cv)=0.288  group_cut=5  gap=1
[bandai] PR-AUC=0.960  ROC-AUC=0.852  F1=0.892  threshold(cv)=0.493  group_cut=5  gap=1
[nikko] PR-AUC=0.828  ROC-AUC=0.752  F1=0.577  threshold(cv)=0.459  group_cut=5  gap=1
[oze] PR-AUC=0.864  ROC-A